In [38]:
import pandas as pd
import numpy as np 
import os
import re

YEAR_FILTER = '0113-2'  # 修改為你想篩選的年份

In [39]:
INPUT_FOLDER = f'processed_data_{YEAR_FILTER}/'  # 資料夾路徑
OUTPUT_FOLDER = f'combine_data_{YEAR_FILTER}/'  # 合併後的輸出資料夾路徑
os.makedirs(OUTPUT_FOLDER, exist_ok=True)  # 建立新資料夾

OUTPUT_OTHER_YEARS = os.path.join(OUTPUT_FOLDER, 'combined_other_years.csv')  # 合併 other_years 資料的輸出檔案
OUTPUT_0113 = os.path.join(OUTPUT_FOLDER, f'combined_{YEAR_FILTER}.csv')  # 合併 0113-1 資料的輸出檔案

# 指定要保留的欄位
REQUIRED_COLUMNS = ['學年期', '系所名稱', '年級班別組別', '類別', '科目名稱(連結課程地圖)             備註 \n             限選條件',
                    '學分選必修', '教師姓名*:主負責老師', '已選課人數/餘額', '時間/教室']

def preprocess_year_and_rank(df, year_filter):
    """
    處理學年期資料，去掉 `-` 並轉換為數值，過濾掉大於 YEAR_FILTER 的資料，
    """
    if '學年期' in df.columns:
        df = df.copy()  # 建立 DataFrame 副本，避免修改原始資料
        # 去掉 `-` 並轉換為數值（例如 '113-1' -> 1131）
        df['學年期'] = df['學年期'].str.replace('-', '').astype(int)
        year_filter_numeric = int(year_filter.replace('-', ''))  # 將 YEAR_FILTER 轉換為數值
        
        # 過濾大於 YEAR_FILTER 的資料
        df = df[df['學年期'] <= year_filter_numeric].copy()
    
    return df

def split_enrollment_and_balance(df):
    """將選課人數/餘額欄位分割成兩個欄位"""
    if '已選課人數/餘額' in df.columns:
        df[['選課人數', '餘額']] = df['已選課人數/餘額'].str.split('/', expand=True)
        df['選課人數'] = pd.to_numeric(df['選課人數'], errors='coerce')  # 將選課人數轉換為數字
        df['餘額'] = pd.to_numeric(df['餘額'], errors='coerce')          # 將餘額轉換為數字

        # 將餘額中的空白或 NaN 值補 0
        df['餘額'] = df['餘額'].fillna(0)
        
        df.drop('已選課人數/餘額', axis=1, inplace=True)  # 刪除原始欄位
    return df

def split_time_and_classroom(df):
    """將時間/教室欄位分割成兩個欄位，並移除教室欄位"""
    if '時間/教室' in df.columns:
        def split_time_class(s):
            for i, char in enumerate(s):
                # 判斷第一個不是 []、~ 或數字的字元位置
                if not (char.isdigit() or char in '[]~'):
                    return s[:i].strip(), s[i:].strip()
            return s.strip(), None  # 無法分割時，返回原字串和 None
        
        split_result = df['時間/教室'].apply(split_time_class)
        df['時間'], _ = zip(*split_result)
        
        # 將時間欄位中的空白或 NaN 值填補為 '未定'
        df['時間'] = df['時間'].fillna('未定')
        df.drop('時間/教室', axis=1, inplace=True)  # 刪除原始欄位
    return df

def split_credits_and_type(df):
    """將學分選必修欄位分割成兩個欄位"""
    if '學分選必修' in df.columns:
        # 使用正則提取學分和選修類型
        df['學分'] = df['學分選必修'].str.extract(r'(\d+)').astype(float)  # 提取數字部分
        df['選必修類型'] = df['學分選必修'].str.extract(r'([必選修]+)')  # 提取必修或選修
        df.drop('學分選必修', axis=1, inplace=True)  # 刪除原始欄位
    return df

def split_grade_and_class(df):
    """將年級班別組別拆分為年級和班別組別兩個欄位，並處理空白或特定情況"""
    if '年級班別組別' in df.columns:
        # 確保欄位為字串格式，並處理空值
        df['年級班別組別'] = df['年級班別組別'].fillna('').astype(str)

        # 提取數字部分作為年級
        df['年級'] = df['年級班別組別'].str.extract(r'^(\d+)')  # 開頭的數字部分

        # 提取剩餘部分作為班別組別
        df['班別組別'] = df['年級班別組別'].str.extract(r'(\D+)$')  # 結尾的非數字部分

        # 將空白或無效的班別組別替換為 '合班'
        df['班別組別'] = df['班別組別'].fillna('合班')  # 填補空值
        df['班別組別'] = df['班別組別'].replace(r'^\s*$', '合班', regex=True)  # 處理僅有空白的情況

        # 將指定的班別組別（如甲乙）替換為 '合班'
        df['班別組別'] = df['班別組別'].replace(['甲乙', '乙甲'], '合班')  # 加入您需要替換的內容

        # 刪除原始欄位
        df.drop('年級班別組別', axis=1, inplace=True)
    return df

def combine_files(input_folder, keyword, output_file, required_columns, year_filter):
    """合併資料夾中包含特定關鍵字的 CSV 檔案，處理學年期排名並進行資料預處理"""
    all_data = []
    for file in os.listdir(input_folder):
        if file.endswith('.csv') and keyword in file:
            file_path = os.path.join(input_folder, file)
            print(f"讀取檔案: {file_path}")
            df = pd.read_csv(file_path)

            # 僅保留指定的欄位
            missing_columns = [col for col in required_columns if col not in df.columns]
            if missing_columns:
                print(f"檔案 {file} 缺少以下必要欄位，跳過處理: {missing_columns}")
                continue

            df = df[required_columns]

            # 預處理學年期，並計算排名
            df = preprocess_year_and_rank(df, year_filter)

            # 分割選課人數/餘額
            df = split_enrollment_and_balance(df)

            # 分割時間/教室
            df = split_time_and_classroom(df)

            # 分割學分與選必修類型
            df = split_credits_and_type(df)

            # 分割年級和班別組別
            df = split_grade_and_class(df)

            # 合併系所名稱與科目名稱到新欄位，並在中間加一個空白
            df['系所科目'] = df['系所名稱'] + ' ' + df['科目名稱(連結課程地圖)             備註 \n             限選條件']

            # 刪除原本的系所名稱與科目名稱欄位
            df.drop(columns=['系所名稱', '科目名稱(連結課程地圖)             備註 \n             限選條件'], inplace=True)

            # 將系所科目欄位移至第二欄
            columns = df.columns.tolist()
            columns.insert(1, columns.pop(columns.index('系所科目')))  # 將 '系所科目' 移到第二欄
            df = df[columns]

            all_data.append(df)

    if all_data:
        combined_df = pd.concat(all_data, ignore_index=True)
        combined_df.to_csv(output_file, index=False, encoding='utf-8')
        print(f"已將包含 '{keyword}' 的檔案合併並儲存至 {output_file}")
    else:
        print(f"未找到包含 '{keyword}' 的檔案進行合併")

# 合併 other_years 和 0113-1 的檔案
combine_files(INPUT_FOLDER, 'other_years', OUTPUT_OTHER_YEARS, REQUIRED_COLUMNS, YEAR_FILTER)
combine_files(INPUT_FOLDER, YEAR_FILTER, OUTPUT_0113, REQUIRED_COLUMNS, YEAR_FILTER)

讀取檔案: processed_data_0113-2/數學系 MATH_other_years.csv
讀取檔案: processed_data_0113-2/測量系 GM_other_years.csv
讀取檔案: processed_data_0113-2/統計系 STAT_other_years.csv
讀取檔案: processed_data_0113-2/資訊系 CSIE_other_years.csv
已將包含 'other_years' 的檔案合併並儲存至 combine_data_0113-2/combined_other_years.csv
讀取檔案: processed_data_0113-2/數學系 MATH_0113-2.csv
讀取檔案: processed_data_0113-2/測量系 GM_0113-2.csv
讀取檔案: processed_data_0113-2/統計系 STAT_0113-2.csv
讀取檔案: processed_data_0113-2/資訊系 CSIE_0113-2.csv
已將包含 '0113-2' 的檔案合併並儲存至 combine_data_0113-2/combined_0113-2.csv


In [40]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, BaggingRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_squared_error

def convert_year_to_rank(df, column_name):
    """將學年期前三個數字轉換為排名，越大排名越高，並將排名移到第一欄，刪除學年期欄位"""
    if column_name not in df.columns:
        raise ValueError(f"資料中缺少 '{column_name}' 欄位")
    # 確保欄位為字串格式
    df[column_name] = df[column_name].astype(str)
    # 提取前三個數字
    df['學年期排名'] = df[column_name].str.extract(r'^(\d{3})').astype(float)
    # 計算排名（越大排名越高，連續排名）
    df['學年期排名'] = df['學年期排名'].rank(method='dense', ascending=False).astype(int)
    # 將學年期排名移到第一欄
    columns = ['學年期排名'] + [col for col in df.columns if col != '學年期排名']
    df = df[columns]
    return df

# 讀取資料
data_path = f'combine_data_{YEAR_FILTER}/combined_other_years.csv'  # 修改為你的檔案路徑
true_path = f'combine_data_{YEAR_FILTER}/combined_{YEAR_FILTER}.csv'
df = pd.read_csv(data_path)
true_df = pd.read_csv(true_path)

# 處理學年期排名
df = convert_year_to_rank(df, '學年期')
true_df = convert_year_to_rank(true_df, '學年期')

df['學年期排名'] = df['學年期排名'] + 1

# 刪除學年期欄位
df.drop('學年期', axis=1, inplace=True)
true_df.drop('學年期', axis=1, inplace=True)

# 確保選課人數與餘額欄位存在並處理缺失值
required_columns = ['選課人數', '餘額']
for col in required_columns:
    if col not in df.columns or col not in true_df.columns:
        raise ValueError(f"資料中缺少 '{col}' 欄位")

# 分離目標變數（選課人數和餘額）和特徵
y1 = df['選課人數']
y2 = df['餘額']

# 分離需要保留為數值型的欄位（學分、年級、學年期排名）
numeric_features = ['學分', '年級', '學年期排名']
if not all(col in df.columns for col in numeric_features) or not all(col in true_df.columns for col in numeric_features):
    raise ValueError("資料中缺少數值型特徵欄位")

numeric_df = df[numeric_features]
numeric_true_df = true_df[numeric_features]

# 移除數值型特徵後的資料
X = df.drop(columns=required_columns + numeric_features)
X_pred = true_df.drop(columns=required_columns + numeric_features)

# One-Hot Encoding 對剩餘特徵欄位處理
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_encoded = encoder.fit_transform(X)
X_pred_encoded = encoder.transform(X_pred)

# 合併數值型特徵與 One-Hot Encoding 後的資料
X_final = np.hstack([numeric_df.to_numpy(), X_encoded])
X_pred_final = np.hstack([numeric_true_df.to_numpy(), X_pred_encoded])

print(f"One-Hot Encoding 完成，最終特徵數量：{X_final.shape[1]}")

# 資料分割
X_train, X_test, y1_train, y1_test = train_test_split(X_final, y1, test_size=0.2, random_state=42)
_, _, y2_train, y2_test = train_test_split(X_final, y2, test_size=0.2, random_state=42)

# 建立 Random Forest Regression 模型
model_y1 = BaggingRegressor(estimator=RandomForestRegressor(n_estimators=100, random_state=42), n_estimators=10, random_state=43)
model_y1.fit(X_train, y1_train)

model_y2 = BaggingRegressor(estimator=RandomForestRegressor(n_estimators=100, random_state=44), n_estimators=10, random_state=45)
model_y2.fit(X_train, y2_train)

# 預測
y1_true_pred = model_y1.predict(X_pred_final)
y2_true_pred = model_y2.predict(X_pred_final)

# 依“系所名稱”分組並儲存
output_folder = f'predicted_results_{YEAR_FILTER}/'  # 輸出資料夾
os.makedirs(output_folder, exist_ok=True)  # 建立資料夾

# 確保“系所科目”欄位存在
if '系所科目' not in true_df.columns:
    raise ValueError("資料中缺少 '系所科目' 欄位")

# 使用正則表達式進行切分，從第二個空白處分為兩部分
def split_after_second_space(row):
    # 使用正則表達式，從第二個空白切割
    parts = re.split(r'(\S+\s+\S+)\s+', row, maxsplit=1)  # 找到第二個空白處後的分割點
    if len(parts) > 2:
        return parts[1], parts[2]  # 前面部分和後面部分
    return row, ''  # 若無法切分，返回原字串和空白

true_df['系所名稱'], true_df['科目名稱'] = zip(*true_df['系所科目'].map(split_after_second_space))

# 建立包含預測結果的 DataFrame
true_df['預測選課'] = y1_true_pred.round(0)
true_df['預測餘額'] = y2_true_pred.round(0)

# 刪除原始“系所科目”欄位
true_df.drop(columns=['系所科目'], inplace=True)

for department, group in true_df.groupby('系所名稱'):
    # 只保留需要的三個欄位
    output_data = group[['科目名稱', '預測選課', '預測餘額']]
    file_path = os.path.join(output_folder, f"{department}_predictions.csv")
    output_data.to_csv(file_path, index=False, encoding='utf-8')
    print(f"已儲存 {department} 的預測結果至 {file_path}")

One-Hot Encoding 完成，最終特徵數量：542
已儲存 數學系 MATH 的預測結果至 predicted_results_0113-2/數學系 MATH_predictions.csv
已儲存 測量系 GM 的預測結果至 predicted_results_0113-2/測量系 GM_predictions.csv
已儲存 統計系 STAT 的預測結果至 predicted_results_0113-2/統計系 STAT_predictions.csv
已儲存 資訊系 CSIE 的預測結果至 predicted_results_0113-2/資訊系 CSIE_predictions.csv


In [41]:
# 指定資料夾路徑
input_folder = f'predicted_results_{YEAR_FILTER}/'  # 修改為您的資料夾路徑

# 獲取資料夾中的所有 .csv 檔案
files = [file for file in os.listdir(input_folder) if file.endswith('.csv')]

for file in files:
    file_path = os.path.join(input_folder, file)
    # 讀取每個檔案
    df = pd.read_csv(file_path)
    
    # 合併相同科目名稱的資料，將預測選課和預測餘額相加
    merged_df = df.groupby('科目名稱', as_index=False).agg({
        '預測選課': 'sum',
        '預測餘額': 'sum'
    })
    
    # 將合併後的資料儲存回檔案
    merged_df.to_csv(file_path, index=False, encoding='utf-8')
    print(f"已合併並更新檔案：{file_path}")

已合併並更新檔案：predicted_results_0113-2/數學系 MATH_predictions.csv
已合併並更新檔案：predicted_results_0113-2/測量系 GM_predictions.csv
已合併並更新檔案：predicted_results_0113-2/統計系 STAT_predictions.csv
已合併並更新檔案：predicted_results_0113-2/資訊系 CSIE_predictions.csv
